# Inspection of nominatim querys
This notebook extracts and plots the polygons of cities from the OSM, using nominatim queries.

## Setup

In [37]:
%run -i "functions.py"
cityfilename = "european_100000pop.csv"
boundarycheck_quick = False

## Load cities

In [40]:
with open('../cities/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=",")
    header = next(reader)
    cities = {slugify(rows[0]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2]} for rows in reader}
os.makedirs("../plots/", exist_ok=True)

## Check city boundaries

In [42]:
CACHE_DIR = "../cache/nominatim"
os.makedirs(CACHE_DIR, exist_ok=True)

def geocode_cached(cityid, query):
    cache_file = os.path.join(CACHE_DIR, f"{cityid}.geojson")
    if os.path.exists(cache_file):
        gdf = gpd.read_file(cache_file)
    else:
        try:
            gdf = ox.geocoder.geocode_to_gdf(query)
        except TypeError:
            return "no_polygon"
        except ox._errors.InsufficientResponseError:
            return "no_results"
        gdf.to_file(cache_file, driver="GeoJSON")
        time.sleep(1)
    return shapely.geometry.shape(gdf['geometry'][0])

In [52]:
cities_with_no_polygon = []
cities_with_multipolygon = []
cities_with_no_results = []

for i, (cityid, city_info) in enumerate(tqdm(cities.items(), desc="Checking city boundaries", position=0, leave=True)):

    output_path = f"../plots/boundary_{cityid}.jpeg"
    if os.path.exists(output_path):
        continue

    # Draw location polygons and their holes
    if city_info["nominatim_query"]:
        location = geocode_cached(cityid, city_info["nominatim_query"])
        if location == "no_polygon":
            cities_with_no_polygon.append(city_info["name_en"])
            continue
        if location == "no_results":
            cities_with_no_results.append(city_info["name_en"])
            continue
        is_poly, geoms, has_holes, polyg = analyse_polygon(location)
    else:
        # https://github.com/mszell/bikenwgrowth/blob/main/code/01_prepare_networks.ipynb
        shp = gpd.read_file("../cities/"+cityid+".shp")
        location = shp.iloc[0].geometry

    if not is_poly:
        cities_with_no_polygon.append(city_info["name_en"])
        continue

    if geoms > 1:
        cities_with_multipolygon.append(city_info["name_en"])

    fig = plt.figure()
    ax = fig.add_axes([0,0,1,1])
    if geoms > 1:
        geom_list = list(location.geoms)
        colors = cm.rainbow(np.linspace(0, 1, len(geom_list)))
        for poly, c in zip(geom_list, colors):
            plt.plot(*poly.exterior.xy, c=c, linewidth=2)
            for intr in poly.interiors:
                plt.plot(*intr.xy, c="red", linewidth=1)
    else:
        plt.plot(*location.exterior.xy, linewidth=2)
        for intr in location.interiors:
            plt.plot(*intr.xy, c="red", linewidth=1)

    # Largest polygon (holes filled) in black
    plt.plot(*polyg.exterior.xy, c="black", linewidth=2)
    
    contextily.add_basemap(ax=ax, url=contextily.providers.CartoDB.Positron, crs='EPSG:4326', alpha = 0.5)
    plt.gca().set_aspect('equal')
    ax.set_title(city_info["name_en"]+", "+city_info["country_en"])
    #plt.show()
    fig.savefig(f"../plots/boundary_{cityid}.jpeg", dpi=150, bbox_inches='tight')
    plt.close()

Checking city boundaries: 100%|██████████| 455/455 [01:31<00:00,  4.97it/s]


## Individual checks

First we check the cities which queries didn't return any result, and change the nominatim queries to something fitting.

Then we check the cities which queries didn't return a polygon, and change the nominatim queries to something fitting.

Finally, we checked each city individually comparing the plot polygon with the equivalent in Google Maps, labelling according the following tags:
- **good**: if the polygon is the same
- **bigger**: if the polygon was bigger, extracting metropolitan areas or whole regions instead of just the municipality
- **different**: if the polygon extracted was from a different location

### Non-capital European cities

#### Cities with no results

In [44]:
# First pass
# Check which cities had no results from the query
cities_with_no_results

['Pésterion', 'Shauliai']

These cities had the wrong name in the UN Demographics Yearbook

- Pésterion = Peristeri (Greece)
- Shauliai = Šiauliai (Lithuania)

In [45]:
# Make the changes
cities["peristeri"] = cities.pop("pesterion")
cities["peristeri"]["name_en"] = "Peristeri"
cities["peristeri"]["nominatim_query"] = "Peristeri"

cities["siauliai"] = cities.pop("shauliai")
cities["siauliai"]["name_en"] = "Šiauliai"
cities["siauliai"]["nominatim_query"] = "Šiauliai"

In [55]:
# Second pass
# Check which cities had no results from the query
cities_with_no_results

[]

#### Cities without a polygon

In [46]:
# First pass
# Check cities where the query returned a geometry but it was not a polygon
cities_with_no_polygon

['Aalborg',
 'Aarhus',
 'Calithèa',
 'Esbjerg',
 'Frederiksberg',
 'Heraklion',
 'Larissa',
 "Luts'K",
 'Odense',
 'Patras',
 'Piraeus',
 'Uzhhorod',
 'Vejle']

These cities had a geometry extracted that wasn't a polygon:

Denmark:
- Aalborg
- Aarhus
- Esbjerg
- Frederiksberg
- Odense
- Vejle

Greece:
- Piraeus
- Heraklion
- Larissa
- Calithèa
- Patras

Ukraine:
- Luts'K
- Uzhhorod

They represent all Danish and Greek cities, and some from Ukraine.

In [ ]:
# For Danish cities, we add "Municipality" in the query
cities['aalborg']['nominatim_query'] = "Aalborg Municipality"
cities['aarhus']['nominatim_query'] = "Aarhus Municipality"
cities['esbjerg']['nominatim_query'] = "Esbjerg Municipality"
cities['frederiksberg']['nominatim_query'] = "Frederiksberg Municipality"
cities['odense']['nominatim_query'] = "Odense Municipality"
cities['vejle']['nominatim_query'] = "Vejle Municipality"


# For Greek cities, we use the Greek name in the query 
cities['piraeus']['nominatim_query'] = "Πειραιάς"
cities['heraklion']['nominatim_query'] = "Ηράκλειο"
cities['larissa']['nominatim_query'] = "Λάρισα"

#cities["kallithea"] = cities.pop("calithea") # The city Kallithea was mispelled from the UN source.
cities["kallithea"]["name_en"] = "Kallithea"
cities["kallithea"]["nominatim_query"] = "Καλλιθέα"

cities['patras']['nominatim_query'] = "Πάτρα"


# For Ukraine we correct the name and use a tested nominatim in OSM's nominatim web interface
cities['luts-k']['nominatim_query'] = "Lutsk" # The city Lutsk was mispelled from the UN source.
cities['uzhhorod']['nominatim_query'] = "Uzhhorod Urban Hromada"

In [54]:
# Second pass
# Check cities where the query returned a geometry but it was not a polygon
cities_with_no_polygon

['Heraklion', 'Larissa', 'Patras']

In [8]:
new_cities = copy.deepcopy(cities)

#### Bigger

In [ ]:
# Draw location polygons and their holes
city = 'belfast'
city_info = cities[city]

new_nominatim_query = "Belfast, United Kingdom"

if city_info["nominatim_query"]:
    location = geocode_cached(cityid, new_nominatim_query)
    if location == "no_polygon":
        cities_with_no_polygon.append(city_info["name_en"])
        continue
    if location == "no_results":
        cities_with_no_results.append(city_info["name_en"])
        continue
    is_poly, geoms, has_holes, polyg = analyse_polygon(location)
else:
    # https://github.com/mszell/bikenwgrowth/blob/main/code/01_prepare_networks.ipynb
    shp = gpd.read_file("../cities/"+cityid+".shp")
    location = shp.iloc[0].geometry

if not is_poly:
    cities_with_no_polygon.append(city_info["name_en"])
    continue

if geoms > 1:
    cities_with_multipolygon.append(city_info["name_en"])

fig = plt.figure()
ax = fig.add_axes([0,0,1,1])
if geoms > 1:
    geom_list = list(location.geoms)
    colors = cm.rainbow(np.linspace(0, 1, len(geom_list)))
    for poly, c in zip(geom_list, colors):
        plt.plot(*poly.exterior.xy, c=c, linewidth=2)
        for intr in poly.interiors:
            plt.plot(*intr.xy, c="red", linewidth=1)
else:
    plt.plot(*location.exterior.xy, linewidth=2)
    for intr in location.interiors:
        plt.plot(*intr.xy, c="red", linewidth=1)

# Largest polygon (holes filled) in black
plt.plot(*polyg.exterior.xy, c="black", linewidth=2)

contextily.add_basemap(ax=ax, url=contextily.providers.CartoDB.Positron, crs='EPSG:4326', alpha = 0.5)
plt.gca().set_aspect('equal')
ax.set_title(city_info["name_en"]+", "+city_info["country_en"])
plt.show()
#fig.savefig(f"../plots/boundary_{cityid}.jpeg", dpi=150, bbox_inches='tight')
plt.close()